# Quem aparece nas notícias: reconhecimento de entidades

*Reconhecimento de entidades* (em inglês, *named entity recognition*, ou NER) é a técnica que encontra nomes próprios num texto e diz de que tipo cada um é. O modelo de português do spaCy reconhece quatro tipos:

| Tipo | O que é | Exemplo |
|---|---|---|
| `PER` | pessoa | Lula |
| `ORG` | organização | Ministério da Saúde |
| `LOC` | lugar | São Paulo |
| `MISC` | outros nomes próprios | nomes de eventos, leis, prêmios |

Neste notebook usamos as notícias coletadas em [03-agencia-brasil.ipynb](03-agencia-brasil.ipynb) para responder: **quais pessoas e instituições aparecem nas notícias da Agência Brasil?** Se as notícias ainda não foram coletadas, este notebook roda o 03 do começo, o que leva uns 10 minutos.

Os exemplos abaixo usam a coleta de 2 de novembro de 2026, que está salva neste notebook. Se você coletou outras notícias, os nomes vão ser outros: troque os nomes dos exemplos pelos que aparecerem na sua coleta.

In [1]:
from pathlib import Path

import pandas as pd
import spacy
from spacy import displacy

CSV = Path('../data/agencia-brasil.csv')
if not CSV.exists():
    # As notícias ainda não foram coletadas: roda o notebook 03 do começo
    %run 03-agencia-brasil.ipynb

news = pd.read_csv(CSV)
nlp = spacy.load('pt_core_news_sm')

## Um exemplo

O `displacy`, do spaCy, marca as entidades no texto, com o tipo de cada uma. Este parágrafo é de uma notícia de política:

In [2]:
def paragraph_with(text):
    for article in news['texto']:
        for paragraph in article.split('\n'):
            if text in paragraph:
                return paragraph

displacy.render(nlp(paragraph_with('Donald Trump')), style='ent', jupyter=True)

## Todas as entidades das notícias

Agora passamos todas as notícias pelo modelo e guardamos cada entidade numa tabela, com a editoria, o link da notícia e a frase em que ela aparece. A frase vai ser útil para conferir os resultados.

In [3]:
rows = []
for (_, article), doc in zip(news.iterrows(), nlp.pipe(news['texto'])):
    for entity in doc.ents:
        rows.append({
            'editoria': article['editoria'],
            'url': article['url'],
            'entidade': entity.text,
            'tipo': entity.label_,
            'frase': entity.sent.text.strip(),
        })

entities = pd.DataFrame(rows)
entities['tipo'].value_counts()

tipo
LOC     621
ORG     305
MISC    280
PER     189
Name: count, dtype: int64

## As pessoas que mais aparecem

Como vimos no notebook 03, uma notícia que repete muito um nome pode distorcer a contagem. Por isso contamos **em quantas notícias** cada nome aparece, e não quantas vezes.

In [4]:
def top(entities, kind, n=10):
    found = entities[entities['tipo'] == kind].drop_duplicates(['url', 'entidade'])
    return found['entidade'].value_counts().head(n).rename('notícias')

top(entities, 'PER')

entidade
Luiz Inácio Lula da Silva           7
Flávio Bolsonaro                    6
Lula                                5
Felipe                              2
Flávio                              2
Jair Bolsonaro                      2
PL                                  2
Enem                                2
Marco Global de Kunming-Montreal    2
Inmet                               2
Name: notícias, dtype: int64

A lista parece boa, mas tem três problemas, e eles aparecem em qualquer análise de entidades.

## Problema 1: a mesma pessoa, com nomes diferentes

"Lula" e "Luiz Inácio Lula da Silva" são a mesma pessoa, mas para o computador são textos diferentes. Com "Flávio" é mais difícil: pode ser Flávio Bolsonaro ou outro Flávio qualquer. Para saber, é preciso ler as frases:

In [5]:
def show_contexts(name, n=5):
    for _, row in entities[entities['entidade'] == name].head(n).iterrows():
        print(f"[{row['editoria']}] {row['frase']}\n")

show_contexts('Flávio')

[politica] Flávio nega qualquer irregularidade.

[politica] Derrotado nas urnas no dia 25 de outubro, Flávio deixou o país após a Polícia Federal pedir o confisco do passaporte do parlamentar por entender que ele poderia fugir em função dos processos que enfrenta na Justiça.

[politica] O placar inspirava preocupação dos estudantes, que viam Flávio como uma ameaça à educação.

[politica] A vitória de Flávio claramente significaria corte em todas as áreas das universidades, da infraestrutura ao auxílio permanência, um roteiro conhecido", avaliou Valentina, em relação ao governo de Jair Bolsonaro, entre 2019 e 2022.



Nessas notícias, "Flávio" é sempre Flávio Bolsonaro. Juntamos os nomes com um dicionário que diz qual é o nome padrão de cada variação. Não existe um jeito automático confiável de fazer isso: quem decide é você, depois de conferir.

In [6]:
ALIASES = {
    'Luiz Inácio Lula da Silva': 'Lula',
    'Flávio': 'Flávio Bolsonaro',
}

entities['entidade'] = entities['entidade'].replace(ALIASES)
top(entities, 'PER')

entidade
Lula                                7
Flávio Bolsonaro                    6
Felipe                              2
Jair Bolsonaro                      2
PL                                  2
Enem                                2
Marco Global de Kunming-Montreal    2
Inmet                               2
Disque                              1
Janine Mello                        1
Name: notícias, dtype: int64

Repare que Lula estava em 7 notícias com o nome completo e em 5 como "Lula", mas juntando dá 7, e não 12: muitas notícias usam as duas formas, e contamos cada notícia uma vez só.

## Problema 2: o mesmo nome, pessoas diferentes

O contrário também acontece. Veja quem é "Felipe":

In [7]:
show_contexts('Felipe')

[politica] Essa foi a mensagem de áudio que Felipe* (nome fictício), de 17 anos, recebeu em um grupo de WhatsApp de estudantes da turma dele em uma escola no Distrito Federal.

[politica] Felipe é um adolescente transgênero e segundo a mãe, Aline* (nome fictício), ele nunca sofreu transfobia na escola.

[politica] Aline também contou que, por causa das agressões, Felipe passou a semana toda sem ir à escola.

[politica] Nós estamos orgulhosos dos estudantes da UFF e dos estudantes de todo o país", declarou Felipe, que faz doutorado na Universidade Federal Fluminense, além de já ter presidido a OAB Nacional e a seccional do Rio de Janeiro.

[politica] Felipe tinha apenas 2 anos quando o pai se tornou um desaparecido político.



São duas pessoas: um adolescente de 17 anos e um advogado que já presidiu a OAB e hoje faz doutorado na UFF. E o nome do adolescente é fictício: a reportagem trocou o nome para protegê-lo, como se faz com menores de idade e fontes em risco. A análise automática não sabe nada disso, e um nome fictício contado como se fosse uma pessoa real pode virar um erro publicado.

## Problema 3: o modelo erra o tipo

Nem tudo o que o modelo chama de pessoa é pessoa, e o mesmo vale para os outros tipos. Veja os lugares:

In [8]:
top(entities, 'LOC')

entidade
Brasil                    20
São Paulo                  8
MEC                        6
Ministério da Educação     5
Pará                       5
Polícia Federal            4
Rio de Janeiro             4
Minas Gerais               4
Paraná                     4
Santa Catarina             4
Name: notícias, dtype: int64

MEC, Ministério da Educação e Polícia Federal não são lugares, são organizações. Na lista de pessoas acima também aparecem PL (um partido) e Enem (um exame). O modelo erra mais com órgãos públicos, siglas e nomes que não viu no treinamento. Neste parágrafo, quase metade dos órgãos foi marcada como lugar (`LOC`), e o mesmo órgão pode ganhar um tipo por extenso e outro na sigla: "Defensoria Pública da União" virou lugar, e "DPU", organização.

In [9]:
displacy.render(nlp(paragraph_with('Defensoria Pública da União')), style='ent', jupyter=True)

## E as organizações?

Os mesmos cuidados valem para as organizações. Aqui a lista também tem a própria Agência Brasil, que aparece quando a notícia diz que procurou alguém ou deu uma entrevista à agência:

In [10]:
top(entities, 'ORG')

entidade
Ministério da Saúde                         6
Agência Brasil                              5
Agência Nacional de Vigilância Sanitária    4
SUS                                         4
Ministério Público do Trabalho              3
MPT                                         3
EBC                                         3
US$                                         3
B3                                          3
Agência Nacional de Energia Elétrica        3
Name: notícias, dtype: int64

## O que encontramos

Na coleta de 2 de novembro de 2026, as pessoas que mais aparecem são da política, logo depois da eleição presidencial: Lula e Flávio Bolsonaro. Entre as organizações, aparecem os órgãos que as notícias de saúde e economia citam como fonte, como o Ministério da Saúde, a Anvisa e a Aneel.

O reconhecimento de entidades é um bom ponto de partida para uma apuração: em segundos, ele mostra quem aparece em centenas de textos. Mas não publique a contagem direto. Antes:

* **Junte as variações do mesmo nome**, conferindo as frases antes de juntar.
* **Separe pessoas diferentes com o mesmo nome**, e cuidado com nomes fictícios.
* **Revise os tipos.** Use o modelo para fazer uma primeira triagem e confira os nomes que vão para a reportagem.